In [3]:
import gridstatus
import pandas as pd

caiso = gridstatus.CAISO()
# ── CORRECTED API CALL ──────────────────────────────────────────────────────

# The correct hub location strings for CAISO trading hubs:
HUB_LOCATIONS = ["TH_NP15_GEN-APND", "TH_SP15_GEN-APND", "TH_ZP26_GEN-APND"]

# Market string — gridstatus CAISO supports these exactly:
# "DAY_AHEAD_HOURLY", "REAL_TIME_HOURLY", "REAL_TIME_15_MIN", "REAL_TIME_5_MIN"
MARKET = "DAY_AHEAD_HOURLY"
MAX_RETRIES=1

def fetch_month(caiso, start: str, end: str, retries: int = MAX_RETRIES) -> pd.DataFrame:
    """Fetch one month of CAISO DA hourly LMP with retry logic."""
    for attempt in range(retries):
        try:
            df = caiso.get_fuel_mix(      # or get_lmp
                date=start,           # <-- 'date' not 'start'
                end=end,
                #market=MARKET,
                #locations=HUB_LOCATIONS,   # <-- 'locations' (list), not 'location_type'
                verbose=False,
            )
            return df
        except Exception as e:
            if attempt < retries - 1:
                print(f"  [retry {attempt+1}/{retries}] {e}. Waiting {RETRY_DELAY}s...")
                time.sleep(RETRY_DELAY)
            else:
                print(f"  [FAILED] {start} → {end}: {e}")
                return pd.DataFrame()

In [4]:
df_fuel = fetch_month(caiso, start="2023-01-01", end="2026-01-01", retries=1)

100%|████████████████████████████████████████████████████████████████████████████████████████████| 1096/1096 [03:36<00:00,  5.06it/s]


In [17]:
df = pd.read_parquet("data/raw/caiso_load_raw.parquet")

In [18]:
df_load

,Time,Interval Start,Interval End,Load
0,2023-01-01 00:00:00-08:00,2023-01-01 00:00:00-08:00,2023-01-01 00:05:00-08:00,20951.0
1,2023-01-01 00:05:00-08:00,2023-01-01 00:05:00-08:00,2023-01-01 00:10:00-08:00,20908.0
2,2023-01-01 00:10:00-08:00,2023-01-01 00:10:00-08:00,2023-01-01 00:15:00-08:00,20892.0
3,2023-01-01 00:15:00-08:00,2023-01-01 00:15:00-08:00,2023-01-01 00:20:00-08:00,20809.0
4,2023-01-01 00:20:00-08:00,2023-01-01 00:20:00-08:00,2023-01-01 00:25:00-08:00,20719.0
...,...,...,...,...
315393,2025-12-31 23:35:00-08:00,2025-12-31 23:35:00-08:00,2025-12-31 23:40:00-08:00,20976.0
315394,2025-12-31 23:40:00-08:00,2025-12-31 23:40:00-08:00,2025-12-31 23:45:00-08:00,20856.0
315395,2025-12-31 23:45:00-08:00,2025-12-31 23:45:00-08:00,2025-12-31 23:50:00-08:00,20810.0
315396,2025-12-31 23:50:00-08:00,2025-12-31 23:50:00-08:00,2025-12-31 23:55:00-08:00,20801.0


In [20]:
df_fuel

,time
0,2023-01-01 08:00:00+00:00
1,2023-01-01 08:05:00+00:00
2,2023-01-01 08:10:00+00:00
3,2023-01-01 08:15:00+00:00
4,2023-01-01 08:20:00+00:00
...,...
315393,2026-01-01 07:35:00+00:00
315394,2026-01-01 07:40:00+00:00
315395,2026-01-01 07:45:00+00:00
315396,2026-01-01 07:50:00+00:00


In [30]:
df_fuel

,time,interval_start,interval_end,solar,wind,geothermal,biomass,biogas,small_hydro,coal,nuclear,natural_gas,large_hydro,batteries,imports,other
0,2023-01-01 08:00:00+00:00,2023-01-01 00:00:00-08:00,2023-01-01 00:05:00-08:00,-37.0,3193.0,905.0,287.0,212.0,191.0,4.0,2245.0,8225.0,1477.0,200.0,6129.0,0.0
1,2023-01-01 08:05:00+00:00,2023-01-01 00:05:00-08:00,2023-01-01 00:10:00-08:00,-37.0,3441.0,904.0,285.0,212.0,186.0,3.0,2244.0,8098.0,1496.0,259.0,5889.0,0.0
2,2023-01-01 08:10:00+00:00,2023-01-01 00:10:00-08:00,2023-01-01 00:15:00-08:00,-37.0,3624.0,904.0,286.0,212.0,186.0,3.0,2246.0,7888.0,1405.0,231.0,5898.0,0.0
3,2023-01-01 08:15:00+00:00,2023-01-01 00:15:00-08:00,2023-01-01 00:20:00-08:00,-37.0,3835.0,904.0,284.0,212.0,186.0,3.0,2245.0,7637.0,1416.0,321.0,5785.0,0.0
4,2023-01-01 08:20:00+00:00,2023-01-01 00:20:00-08:00,2023-01-01 00:25:00-08:00,-35.0,4012.0,904.0,285.0,213.0,186.0,3.0,2244.0,7378.0,1415.0,350.0,5771.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315393,2026-01-01 07:35:00+00:00,2025-12-31 23:35:00-08:00,2025-12-31 23:40:00-08:00,-47.0,2262.0,768.0,224.0,171.0,268.0,0.0,2260.0,5628.0,2421.0,-484.0,8148.0,0.0
315394,2026-01-01 07:40:00+00:00,2025-12-31 23:40:00-08:00,2025-12-31 23:45:00-08:00,-47.0,2228.0,768.0,223.0,171.0,269.0,0.0,2261.0,5372.0,2405.0,-330.0,8202.0,0.0
315395,2026-01-01 07:45:00+00:00,2025-12-31 23:45:00-08:00,2025-12-31 23:50:00-08:00,-47.0,2207.0,768.0,223.0,172.0,269.0,0.0,2261.0,5319.0,2552.0,-349.0,8100.0,0.0
315396,2026-01-01 07:50:00+00:00,2025-12-31 23:50:00-08:00,2025-12-31 23:55:00-08:00,-47.0,2207.0,768.0,223.0,172.0,269.0,0.0,2261.0,5389.0,2609.0,-547.0,8115.0,0.0


In [23]:
df_load = pd.read_parquet("data/raw/caiso_fuel_mix_raw.parquet")
TARGET_TZ = "UTC"
df_load.columns = [c.strip().lower().replace(" ", "_") for c in df_load.columns]

df_load["time"] = df_load["time"].dt.tz_convert(TARGET_TZ)
print("Converted times")


#keep = ["time", "load"]
#print("Keeping columns")
#available = [c for c in keep if c in df_load.columns]
#print("Available", available)
#df_load = df_load[available]

# ── Numeric coercion ─────────────────────────────────────────────────────
print("Numeric coercion")
for col in ["load"]:
    if col in df_load.columns:
        df_load[col] = pd.to_numeric(df_load[col], errors="coerce")




Converted times
Numeric coercion


In [31]:
df_load = df_load.set_index('time')
load_hourly = df_load.resample("H").mean()

/var/folders/6d/bhrf8zb97pv7w8l_1wf7prc80000gn/T/ipykernel_36191/1502253140.py:2: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  load_hourly = df_load.resample("H").mean()


In [33]:
df_fuel = df_fuel.set_index('time')
fuel_hourly = df_fuel.resample("H").mean()

/var/folders/6d/bhrf8zb97pv7w8l_1wf7prc80000gn/T/ipykernel_36191/1932162008.py:2: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  fuel_hourly = df_fuel.resample("H").mean()


In [34]:
# Load data to UTC
#load_hourly.index = pd.to_datetime(df_load.index).dt.tz_convert('UTC')
#df_load = df_load.set_index('time')
#load_hourly = df_load.resample("H").mean()

load_hourly = load_hourly.rename(columns={'Load': 'caiso_load_mw'})

df_merged = pd.merge(cleaned_sorted, load_hourly, on='time').merge(fuel_hourly, on='time')

In [87]:
cleaned_sorted.columns

Index(['time', 'location', 'lmp', 'energy', 'congestion', 'loss'], dtype='object')

In [35]:
df_merged

,time,location,lmp,energy,congestion,loss,load,interval_start,interval_end,solar,...,biomass,biogas,small_hydro,coal,nuclear,natural_gas,large_hydro,batteries,imports,other
0,2023-01-09 08:00:00+00:00,TH_NP15_GEN-APND,136.59256,143.88766,0.00000,-7.29510,20224.333333,2023-01-09 00:27:30-08:00,2023-01-09 00:32:30-08:00,-45.916667,...,207.333333,221.000000,209.666667,3.083333,2255.916667,8490.166667,1923.916667,-194.083333,4864.083333,0.0
1,2023-01-09 08:00:00+00:00,TH_SP15_GEN-APND,143.25455,143.88766,0.00000,-0.63311,20224.333333,2023-01-09 00:27:30-08:00,2023-01-09 00:32:30-08:00,-45.916667,...,207.333333,221.000000,209.666667,3.083333,2255.916667,8490.166667,1923.916667,-194.083333,4864.083333,0.0
2,2023-01-09 08:00:00+00:00,TH_ZP26_GEN-APND,137.26883,143.88766,0.00000,-6.61883,20224.333333,2023-01-09 00:27:30-08:00,2023-01-09 00:32:30-08:00,-45.916667,...,207.333333,221.000000,209.666667,3.083333,2255.916667,8490.166667,1923.916667,-194.083333,4864.083333,0.0
3,2023-01-09 09:00:00+00:00,TH_NP15_GEN-APND,134.78413,141.25354,0.00000,-6.46941,19572.750000,2023-01-09 01:27:30-08:00,2023-01-09 01:32:30-08:00,-45.666667,...,206.833333,220.916667,217.000000,3.000000,2256.000000,8783.916667,1602.000000,-441.666667,4883.000000,0.0
4,2023-01-09 09:00:00+00:00,TH_SP15_GEN-APND,140.34950,141.25354,0.00000,-0.90402,19572.750000,2023-01-09 01:27:30-08:00,2023-01-09 01:32:30-08:00,-45.666667,...,206.833333,220.916667,217.000000,3.000000,2256.000000,8783.916667,1602.000000,-441.666667,4883.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66739,2026-01-01 06:00:00+00:00,TH_SP15_GEN-APND,36.77782,38.68429,-1.02060,-0.88587,22129.750000,2025-12-31 22:27:30-08:00,2025-12-31 22:32:30-08:00,-47.000000,...,224.750000,171.750000,264.250000,0.000000,2259.666667,6427.250000,2636.833333,-260.583333,8172.250000,0.0
66740,2026-01-01 06:00:00+00:00,TH_ZP26_GEN-APND,37.05816,38.68429,-0.29926,-1.32687,22129.750000,2025-12-31 22:27:30-08:00,2025-12-31 22:32:30-08:00,-47.000000,...,224.750000,171.750000,264.250000,0.000000,2259.666667,6427.250000,2636.833333,-260.583333,8172.250000,0.0
66741,2026-01-01 07:00:00+00:00,TH_NP15_GEN-APND,36.55491,38.29720,-0.19125,-1.55104,21130.583333,2025-12-31 23:27:30-08:00,2025-12-31 23:32:30-08:00,-47.000000,...,223.416667,171.416667,267.416667,0.000000,2260.583333,5819.916667,2458.416667,-467.250000,8050.083333,0.0
66742,2026-01-01 07:00:00+00:00,TH_SP15_GEN-APND,37.07846,38.29720,-0.81280,-0.40595,21130.583333,2025-12-31 23:27:30-08:00,2025-12-31 23:32:30-08:00,-47.000000,...,223.416667,171.416667,267.416667,0.000000,2260.583333,5819.916667,2458.416667,-467.250000,8050.083333,0.0


In [8]:
raw_path = "data/raw/caiso_fuel_mix_raw.parquet"
df_fuel.to_parquet(raw_path, index=False) 
print(f"\\n✓ Raw data saved: {raw_path} ({len(df_fuel):,} rows, {len(df_fuel)})")

\n✓ Raw data saved: data/raw/caiso_fuel_mix_raw.parquet (315,398 rows, 315398)


In [12]:
import pandas as pd
df = pd.read_parquet("data/raw/caiso_lmp_raw.parquet")

In [13]:
#TARGET_TZ    = "US/Pacific"
TARGET_TZ = "UTC"
def clean_and_standardize(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize column names, timezones, and types.

    gridstatus CAISO returns columns:
        Time, Market, Location, LMP, Energy, Congestion, Loss
    Time is timezone-aware (UTC or US/Pacific depending on version).
    """
    if df.empty:
        return df

    df = df.copy()

    # ── Normalize column names ───────────────────────────────────────────────
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

    # ── Timezone: normalize everything to US/Pacific, then strip tz ─────────
    if "time" not in df.columns:
        raise ValueError(f"No 'time' column found. Got: {list(df.columns)}")
    print("TIME", df["time"])
    if df["time"].dtype == object:
        df["time"] = pd.to_datetime(df["time"], utc=True)

    if df["time"].dt.tz is None:
        # Assume UTC if naive
        print("Assuming UTC")
        df["time"] = df["time"].dt.tz_localize("UTC")
    else:
        print("Time zone is ", df["time"].dt.tz)

    df["time"] = df["time"].dt.tz_convert(TARGET_TZ)
    print("Converted times")
    # Keep tz-aware — helpful for DST-safe joins with ERA5
    # (ERA5 UTC timestamps can be converted at join time)

    # ── Keep only relevant columns ───────────────────────────────────────────
    keep = ["time", "location", "lmp", "energy", "congestion", "loss"]
    print("Keeping columns")
    available = [c for c in keep if c in df.columns]
    print("Available", available)
    df = df[available]

    # ── Numeric coercion ─────────────────────────────────────────────────────
    print("Numeric coercion")
    for col in ["lmp", "energy", "congestion", "loss"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # ── Location standardisation ─────────────────────────────────────────────
    print("Location standardization")
    df["location"] = df["location"].str.upper().str.strip()

    return df

In [14]:
#orig_df = df
cleaned = clean_and_standardize(df)

TIME 0       2023-01-09 00:00:00-08:00
1       2023-01-09 00:00:00-08:00
2       2023-01-09 00:00:00-08:00
3       2023-01-09 01:00:00-08:00
4       2023-01-09 01:00:00-08:00
                   ...           
66739   2025-09-30 22:00:00-07:00
66740   2025-09-30 22:00:00-07:00
66741   2025-09-30 23:00:00-07:00
66742   2025-09-30 23:00:00-07:00
66743   2025-09-30 23:00:00-07:00
Name: time, Length: 66744, dtype: datetime64[ns, US/Pacific]
Time zone is  US/Pacific
Converted times
Keeping columns
Available ['time', 'location', 'lmp', 'energy', 'congestion', 'loss']
Numeric coercion
Location standardization


In [15]:
cleaned_sorted = cleaned.sort_values(["time", "location"]).reset_index(drop=True)
cleaned_sorted = cleaned_sorted.drop_duplicates(subset=["time", "location"])

In [41]:
df_fuel.columns

Index(['interval_start', 'interval_end', 'solar', 'wind', 'geothermal',
       'biomass', 'biogas', 'small_hydro', 'coal', 'nuclear', 'natural_gas',
       'large_hydro', 'batteries', 'imports', 'other'],
      dtype='object')

In [42]:
# ── WIDE FORM (for modelling) ────────────────────────────────────────────────

def make_wide(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pivot to one row per timestamp, columns like:
        LMP_NP15, Energy_NP15, Congestion_NP15, Loss_NP15,
        LMP_SP15, Energy_SP15, Congestion_SP15, Loss_SP15,
        LMP_ZP26, Energy_ZP26, Congestion_ZP26, Loss_ZP26

    Also adds derived features:
        Congestion_Spread  = Congestion_SP15 - Congestion_NP15  (Path 26 proxy)
        LMP_Spread         = LMP_SP15 - LMP_NP15
        hour_of_day, day_of_week, month, is_weekend
    """
    components = ["lmp", "energy", "congestion", "loss", "load", "solar", "wind"]
    available  = [c for c in components if c in df.columns]

    wide = df.pivot_table(
        index="time",
        columns="location",
        values=available,
        aggfunc="mean",      # handles rare duplicates after dedup
    )

    # Flatten MultiIndex columns: (lmp, NP15) → LMP_NP15
    wide.columns = [f"{comp.capitalize()}_{loc}" for comp, loc in wide.columns]
    wide = wide.reset_index()

    # ── Derived spread features (key target for congestion modelling) ────────
    if "Congestion_SP15" in wide.columns and "Congestion_NP15" in wide.columns:
        wide["Congestion_Spread_SP15_NP15"] = wide["Congestion_SP15"] - wide["Congestion_NP15"]

    if "LMP_SP15" in wide.columns and "LMP_NP15" in wide.columns:
        wide["LMP_Spread_SP15_NP15"] = wide["LMP_SP15"] - wide["LMP_NP15"]

    # ── Calendar features ────────────────────────────────────────────────────
    t = wide["time"]
    wide["hour_of_day"]  = t.dt.hour
    wide["day_of_week"]  = t.dt.dayofweek        # 0=Mon, 6=Sun
    wide["month"]        = t.dt.month
    wide["year"]         = t.dt.year
    wide["is_weekend"]   = (t.dt.dayofweek >= 5).astype(int)
    wide["season"]       = t.dt.month.map(
        {12: "winter", 1: "winter", 2: "winter",
          3: "spring", 4: "spring", 5: "spring",
          6: "summer", 7: "summer", 8: "summer",
          9: "fall",  10: "fall",  11: "fall"}
    )

    return wide

In [36]:
loc_map = {
    "TH_NP15_GEN-APND": "NP15",
    "TH_SP15_GEN-APND": "SP15", 
    "TH_ZP26_GEN-APND": "ZP26",
}
if df_merged["location"].str.contains("GEN-APND").any():
    df_merged["location"] = df_merged["location"].map(loc_map)

In [39]:
df_merged.to_parquet("data/processed/lmp_load_fuel_merged.parquet")

In [43]:
wide_df = make_wide(df_merged)

In [45]:
wide_df_prev = pd.read_parquet("data/processed/caiso_lmp_wide_with_load.parquet")

In [47]:
len(wide_df_prev.columns)

23

In [48]:
len(wide_df.columns)

29

In [49]:
wide_df.to_parquet("data/processed/caiso_lmp_wide_with_load_and_fuel.parquet")

In [50]:
def make_summary(df_wide: pd.DataFrame) -> pd.DataFrame:
    """Quick stats table for sanity-checking the pull."""
    numeric_cols = df_wide.select_dtypes(include=[np.number]).columns.tolist()
    # Drop calendar features from stats
    exclude = ["hour_of_day", "day_of_week", "month", "year", "is_weekend"]
    stat_cols = [c for c in numeric_cols if c not in exclude]

    stats = df_wide[stat_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
    stats["null_count"] = df_wide[stat_cols].isnull().sum()
    stats["null_pct"]   = (stats["null_count"] / len(df_wide) * 100).round(2)
    return stats

In [51]:
import numpy as np
summary = make_summary(wide_df)
summary

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,null_count,null_pct
Congestion_NP15,22248.0,1.394535,8.565489,-245.859090,-15.123092,-3.823410,-0.535625,0.000000,0.660590,15.729844,27.281635,74.506810,0,0.00
Congestion_SP15,22248.0,-2.635911,7.020923,-72.366500,-24.704011,-14.854218,-3.505147,-0.513245,0.000000,0.626848,10.119285,193.603320,0,0.00
Congestion_ZP26,22248.0,-3.693729,8.492575,-289.548300,-31.629918,-17.010041,-4.400887,-0.258245,0.000000,0.063512,1.039375,16.426340,0,0.00
Energy_NP15,22248.0,47.215003,36.103144,-40.987440,-11.298075,5.696982,30.814678,43.556000,56.381397,101.185548,180.851303,1215.389900,0,0.00
Energy_SP15,22248.0,47.215003,36.103144,-40.987440,-11.298075,5.696982,30.814678,43.556000,56.381397,101.185548,180.851303,1215.389900,0,0.00
Energy_ZP26,22248.0,47.215003,36.103144,-40.987440,-11.298075,5.696982,30.814678,43.556000,56.381397,101.185548,180.851303,1215.389900,0,0.00
Lmp_NP15,22248.0,47.698390,32.637967,-31.707580,-6.611470,10.607641,32.954392,43.442305,55.400500,97.046825,173.930827,1090.904500,0,0.00
Lmp_SP15,22248.0,42.942312,38.370545,-50.526010,-21.292744,-3.357865,25.804243,40.770805,53.093505,98.788005,179.792518,1247.633500,0,0.00
Lmp_ZP26,22248.0,41.440545,34.686742,-48.000000,-23.298577,-4.610005,25.743415,40.611215,52.373310,91.644357,166.059857,1092.808500,0,0.00
Load_NP15,22230.0,24504.391708,4725.320435,11357.916667,16056.612500,18742.216667,21293.895833,23719.750000,26464.291667,34501.962500,39832.963333,47325.416667,18,0.08


In [105]:
summary.to_csv("data/processed/caiso_lmp_summary.csv")

In [59]:
print("\nQuick stats (LMP + Congestion columns):")
display_cols = [c for c in summary.index if "LMP" in c or "Congestion" in c or "Spread" in c]
print(summary.loc[display_cols, ["mean", "std", "min", "50%", "max", "null_pct"]].to_string())


Quick stats (LMP + Congestion columns):
                                 mean        std        min       50%        max  null_pct
Congestion_NP15              1.394535   8.565489 -245.85909  0.000000   74.50681       0.0
Congestion_SP15             -2.635911   7.020923  -72.36650 -0.513245  193.60332       0.0
Congestion_ZP26             -3.693729   8.492575 -289.54830 -0.258245   16.42634       0.0
Congestion_Spread_SP15_NP15 -4.030446  15.127356 -145.98445 -0.207230  439.46241       0.0


In [30]:
display_cols

['Congestion_NP15',
 'Congestion_SP15',
 'Congestion_ZP26',
 'Congestion_Spread_SP15_NP15']

In [31]:
print("\nQuick stats (LMP + Congestion columns):")
display_cols = [c for c in summary.index if "LMP" in c or "Congestion" in c or "Spread" in c]
print(summary.loc[display_cols, ["mean", "std", "min", "50%", "max", "null_pct"]].to_string())


Quick stats (LMP + Congestion columns):
                                 mean        std        min       50%        max  null_pct
Congestion_NP15              1.394535   8.565489 -245.85909  0.000000   74.50681       0.0
Congestion_SP15             -2.635911   7.020923  -72.36650 -0.513245  193.60332       0.0
Congestion_ZP26             -3.693729   8.492575 -289.54830 -0.258245   16.42634       0.0
Congestion_Spread_SP15_NP15 -4.030446  15.127356 -145.98445 -0.207230  439.46241       0.0


In [47]:
wide_df.columns

Index(['time', 'Congestion_NP15', 'Congestion_SP15', 'Congestion_ZP26',
       'Energy_NP15', 'Energy_SP15', 'Energy_ZP26', 'Lmp_NP15', 'Lmp_SP15',
       'Lmp_ZP26', 'Loss_NP15', 'Loss_SP15', 'Loss_ZP26', 'hour_of_day',
       'day_of_week', 'month', 'year', 'is_weekend', 'season'],
      dtype='object')

In [59]:
wide_df["month"].unique()

array([ 1,  2,  3,  4, 10, 11, 12], dtype=int32)

In [56]:
spread_col = "Congestion_Spread_SP15_NP15"
df_tmp = wide_df[["time", spread_col, "month"]].dropna()
months = sorted(df_tmp["month"].unique())
months

[np.int32(1),
 np.int32(2),
 np.int32(3),
 np.int32(4),
 np.int32(10),
 np.int32(11),
 np.int32(12)]

In [62]:
start_date = "2023-01-01"
end_date = "2026-01-01"
months = pd.date_range(
    pd.Timestamp(start_date).replace(day=1),
    pd.Timestamp(end_date) + pd.offsets.MonthEnd(0),
    freq="MS",
)
target_months = [1, 2, 3, 4, 10, 11, 12]
months = months[months.month.isin(target_months)]

In [63]:
months

DatetimeIndex(['2023-01-01', '2023-02-01', '2023-03-01', '2023-04-01',
               '2023-10-01', '2023-11-01', '2023-12-01', '2024-01-01',
               '2024-02-01', '2024-03-01', '2024-04-01', '2024-10-01',
               '2024-11-01', '2024-12-01', '2025-01-01', '2025-02-01',
               '2025-03-01', '2025-04-01', '2025-10-01', '2025-11-01',
               '2025-12-01', '2026-01-01'],
              dtype='datetime64[ns]', freq=None)